### Теперь займемся предобработкой нашего датасета

Импортируем необходимые библиотеки:

In [21]:
import pandas as pd
import numpy as np

Далее, прочтем все имеющиеся датасеты для дальйнешей их обработки и конечного мерджа

In [22]:
df_parsed = pd.read_excel("../../data/df_parsed.xlsx")

In [23]:
df_games_final = pd.read_csv("../../data/df_games_final.csv")
df_popular_final = pd.read_csv("../../data/df_popular_final.csv")
df_web_final = pd.read_csv("../../data/df_web_final.csv")
igdb_steam_games_final = pd.read_csv("../../data/igdb_steam_games_final.csv")

print(df_parsed.columns)
print(df_games_final.columns)
print(df_popular_final.columns)
print(df_web_final.columns)
print(igdb_steam_games_final.columns)

Index(['steam_id', 'game_name', 'game_description_snippet', 'game_price',
       'found_game_price', 'all_language_reviews_type',
       'all_language_reviews_count', 'has_russian_reviews',
       'all_russian_reviews_type', 'all_russian_reviews_count',
       'steam_app_url', 'support_ru_region'],
      dtype='object')
Index(['id', 'first_release_date', 'genres', 'name', 'rating', 'rating_count',
       'total_rating', 'total_rating_count'],
      dtype='object')
Index(['id', 'game_id', 'value', 'external_popularity_source'], dtype='object')
Index(['id', 'game', 'type'], dtype='object')
Index(['uid', 'name', 'id', 'game'], dtype='object')


Для начала сделаем merge нашей основной таблицы df_parsed с igdb_steam_games_final. Поскольку в будущем, нам нужно будет делать merge с таблицами для обогащения, полученных с api запросов (df_games_finalб df_popular_final, df_web_final). Нам нужно добавить id игр с igbd в df_parsed, чтобы в дальнейшем осуществить merge с остальными. 

Проверим пустые значения, уникальные значения и дубли перед мерджем

In [24]:
print(df_parsed["steam_id"].isna().sum())
print(df_parsed["steam_id"].nunique())
print(df_parsed.duplicated(subset=["steam_id"]).sum())

print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
26853
0
0
25701
1


In [25]:
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,id,game
45,1438480,Saviorless,2451784.0,25338
46,1438480,Saviorless,2048921.0,25338


In [26]:
igdb_steam_games_final = igdb_steam_games_final.drop(columns = {"id"})

In [27]:
igdb_steam_games_final

,uid,name,game
0,1313940,My Holiday 2,150866
1,1902870,Vertical Kingdom,202602
2,1071940,Streamer's Life,124447
3,1706570,Chupacabras: Night Hunt,163904
4,1085150,Golf Defied,118634
...,...,...,...
25697,1547900,Active Mummy,156747
25698,2238360,You Have No Time,244854
25699,1490030,Cyber Dodge,159989
25700,1855190,Vektor Z,186672


Нашли дубликат, проверили, эта одна и та же игра с одинаковыми id на igdb и steam, разные только id, оставшийся с 
таблицы(при сохранении не прописали index = False). Его дропнули по причине ненадобности, оставшиеся данные нужны,
по ним будем осуществлять первый мердж (по uid (steam id)), и в дальнейшем по game(igdb id). 

Удалим дубликат:

In [28]:
igdb_steam_games_final = igdb_steam_games_final.drop_duplicates()
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,game


In [29]:
print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
25701
0


Отлично, приступаем в merge

In [30]:
df_parsed_1 =df_parsed.merge(igdb_steam_games_final, left_on="steam_id", right_on="uid",how="inner")
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,uid,name,game
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,1313940,My Holiday 2,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,1902870,Vertical Kingdom,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,1071940,Streamer's Life,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,1706570,Chupacabras: Night Hunt,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,1085150,Golf Defied,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,1547900,Active Mummy,156747
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,2238360,You Have No Time,244854
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,1490030,Cyber Dodge,159989
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,1855190,Vektor Z,186672


Видим датасет получившийся, нужно убрать uid (дублирует steam_id), name (дублирует game_name), и для наглядности переименуем game в igdb_id

In [31]:
df_parsed_1 = df_parsed_1.drop(columns = {"name", "uid"})
df_parsed_1 = df_parsed_1.rename(columns = {"game": "igdb_id"})
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672


Перейдем к следующему датасету (df_web_final) и присоединим к общему колонку type, которая в свою очередь показывае, на какой платформе или каком сайте у игры есть страница.

In [32]:
print(df_web_final.isna().sum())
print(df_web_final.duplicated().sum()) #видим 10 дублей, посмотрим их

id      0
game    0
type    0
dtype: int64
12


In [33]:
df_web_final[df_web_final.duplicated(keep=False)] 


,id,game,type
3064,390579,75067,Twitch
3084,92133,75067,Official Website
3237,979505,75067,YouTube
3238,979504,75067,Twitter
4076,464053,159361,Twitch
10061,800116,33027,Nintendo
10190,960612,33027,Twitter
10532,170916,138871,GOG
10533,159645,138871,Community Wiki
10534,155929,138871,Steam


In [34]:
df_web_final = df_web_final.drop_duplicates()
print(df_web_final[df_web_final.duplicated(keep=False)])
df_web_final.info()

Empty DataFrame
Columns: [id, game, type]
Index: []
<class 'pandas.core.frame.DataFrame'>
Index: 25988 entries, 0 to 25999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      25988 non-null  int64 
 1   game    25988 non-null  int64 
 2   type    25988 non-null  object
dtypes: int64(2), object(1)
memory usage: 812.1+ KB


Есть проблема, в датасете оказалось так, что у одной игры несколько таких площадок и айди дублируются, но с разными площадками, и при попытке смерджить, добавится много ненужных строк, поэтому нужно сначала аггрегировать, датасет уменьшится, и получатся много пропусков в мердженном датасете. Тем не менее, на данном этапе, мы с этим уже ничего не сделаем. Соединим по айди все площадки, и запишем их через запятую, далее можно мердж

In [35]:
df_web = (df_web_final[["game", "type"]].drop_duplicates(["game","type"]).groupby("game", as_index=False)["type"].agg(", ".join))
df_web

,game,type
0,111,Wikipedia
1,527,"YouTube, Facebook"
2,671,Official Website
3,857,Wikipedia
4,858,"Wikipedia, Community Wiki, GOG"
...,...,...
12567,393467,Steam
12568,393587,"Steam, Twitter"
12569,395919,Steam
12570,397479,Steam


In [36]:
df_parsed_2 =df_parsed_1.merge(df_web, left_on="igdb_id", right_on="game",how="left")
df_parsed_2 = df_parsed_2.drop(columns= {"game"})
df_parsed_2

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,Official Website
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo"
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,Steam
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,Steam
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,"Twitter, Discord, YouTube, Official Website"
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,Steam
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,"Discord, Official Website, Steam"


In [37]:
df_parsed_2.isna().sum()

steam_id                          0
game_name                         0
game_description_snippet        522
game_price                     5896
found_game_price                954
all_language_reviews_type      1320
all_language_reviews_count     1320
has_russian_reviews             954
all_russian_reviews_type      24167
all_russian_reviews_count     24167
steam_app_url                   954
support_ru_region               954
igdb_id                           0
type                          13116
dtype: int64

Перейдем к следующему датасету

In [38]:
df_games_final.info()
df_games_final

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25698 entries, 0 to 25697
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  25698 non-null  int64  
 1   first_release_date  25364 non-null  float64
 2   genres              25531 non-null  object 
 3   name                25698 non-null  object 
 4   rating              10120 non-null  float64
 5   rating_count        10120 non-null  float64
 6   total_rating        10554 non-null  float64
 7   total_rating_count  10554 non-null  float64
dtypes: float64(5), int64(1), object(2)
memory usage: 1.6+ MB


,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,1.386115e+09,"Puzzle, Indie",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,1.399334e+09,"Fighting, Sport, Indie, Arcade",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,1.640822e+09,"Puzzle, Strategy, Indie, Card & Board Game",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,1.439165e+09,"Simulator, Adventure, Indie",One Final Breath,NaN,NaN,NaN,NaN
4,13359,1.445299e+09,"Adventure, Indie",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,1.736381e+09,"Role-playing (RPG), Indie",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,1.734048e+09,"Puzzle, Simulator, Indie",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,1.741392e+09,"Puzzle, Adventure, Indie",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,1.568160e+09,Indie,Ain Dodo,NaN,NaN,NaN,NaN


Зная, из документации IGDB, что тип данных у first_release_year - это Unix Time Stamp, модифицируем дату 

In [39]:
df_games_final["first_release_date"] = pd.to_datetime(df_games_final["first_release_date"], unit= "s").dt.date
df_games_final #норм, мерджим пока что

,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,2013-12-04,"Puzzle, Indie",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,2014-05-06,"Fighting, Sport, Indie, Arcade",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,2021-12-30,"Puzzle, Strategy, Indie, Card & Board Game",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,2015-08-10,"Simulator, Adventure, Indie",One Final Breath,NaN,NaN,NaN,NaN
4,13359,2015-10-20,"Adventure, Indie",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,2025-01-09,"Role-playing (RPG), Indie",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,2024-12-13,"Puzzle, Simulator, Indie",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,2025-03-08,"Puzzle, Adventure, Indie",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,2019-09-11,Indie,Ain Dodo,NaN,NaN,NaN,NaN


In [40]:
df_parsed_3 =df_parsed_2.merge(df_games_final, left_on="igdb_id", right_on="id",how="left")
df_parsed_3 = df_parsed_3.drop(columns= {"id", "name"})
df_parsed_3

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,Official Website,2020-10-05,Indie,NaN,NaN,NaN,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo",2024-04-15,"Puzzle, Strategy, Indie, Card & Board Game",NaN,NaN,NaN,NaN
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,Steam,2019-07-31,"Role-playing (RPG), Simulator, Indie",NaN,NaN,NaN,NaN
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN,2021-08-20,"Shooter, Indie",NaN,NaN,NaN,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN,2019-03-11,"Simulator, Sport, Indie",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25734,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,Steam,2021-02-28,"Sport, Adventure, Indie",NaN,NaN,NaN,NaN
25735,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,"Twitter, Discord, YouTube, Official Website",2023-11-09,"Puzzle, Indie",NaN,NaN,NaN,NaN
25736,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,Steam,2021-02-14,Indie,NaN,NaN,NaN,NaN
25737,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,"Discord, Official Website, Steam",2022-01-14,"Shooter, Indie, Arcade",NaN,NaN,NaN,NaN
